# Master table check

Reads the master listing table, displays its first 30 columns, and verifies the joins that produced it. The 59 checks test row grain, join fan-out, source coverage, key agreement and copied values.

**Project step:** §4.2 step 1.0.

**Inputs:** `data/master_listing.parquet`, `data/master_columns.csv`, `data/accolade.duckdb`.

**Outputs:** `outputs/01_phase_01_master_table_checks.csv`.

**Run after:** `preprocessing/build_duckdb.py` and `preprocessing/build_master.py`.

**Runtime:** ~5 min.


In [1]:
# ---- Data ----
INPUT = "data/master_listing.parquet"   # relative to the repo root
OUTPUT_DIR = "outputs"                  # every CSV / XLSX / PNG goes here

# ---- Filters (None = no filter) ----
CITY = None             # "chicago", "dfw", "dc", or a list of them       -> column `market`
NEIGHBORHOOD = None     # submarket/list -> column `market_building__submarket`
START_DATE = None       # "YYYY-MM-DD": keep listings with DATE_COLUMN >= START_DATE
END_DATE = None         # "YYYY-MM-DD": keep listings with DATE_COLUMN <= END_DATE
DATE_COLUMN = "market_unit__enter_market"


In [2]:
from pathlib import Path

import csv
import duckdb
import pandas as pd

BLUE, GREEN, YELLOW, RED, BOLD, RESET = "\033[94m", "\033[92m", "\033[93m", "\033[91m", "\033[1m", "\033[0m"


def step(msg):
    print(f"\n{BOLD}{BLUE}▶ {msg}{RESET}")


def done(msg):
    print(f"{GREEN}  ✓ {msg}{RESET}")


def note(msg):
    print(f"{YELLOW}  • {msg}{RESET}")


ROOT = Path.cwd()
while not (ROOT / "data" / "master_listing.parquet").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
MASTER = ROOT / INPUT
CATALOG = ROOT / "data" / "master_columns.csv"
OUTPUTS = ROOT / OUTPUT_DIR
OUTPUTS.mkdir(parents=True, exist_ok=True)
NN = 1
FILTER_COLUMNS = ["market", "market_building__submarket", DATE_COLUMN]


def _as_list(value):
    """Normalize one filter value or a list of values."""
    if value is None:
        return None
    return [value] if isinstance(value, str) else list(value)


def suffix():
    """Build the shared output suffix from the active filters."""
    parts = []
    if CITY is not None:
        parts.append("+".join(map(str, _as_list(CITY))))
    if NEIGHBORHOOD is not None:
        parts.append("+".join(map(str, _as_list(NEIGHBORHOOD))))
    if START_DATE is not None or END_DATE is not None:
        parts.append(f"{START_DATE or 'start'}_to_{END_DATE or 'end'}")
    return "" if not parts else "__" + "__".join(parts)


def out(name, ext="csv"):
    """Build an output path with this notebook's number and active filters."""
    return OUTPUTS / f"{NN:02d}_phase_01_{name}{suffix()}.{ext}"


def upstream(number, name, ext="csv"):
    """Build an upstream notebook's output path with the active filters."""
    return OUTPUTS / f"{number:02d}_phase_01_{name}{suffix()}.{ext}"


def need(path, notebook):
    """Raise a clear instruction when a required upstream output is missing."""
    if not Path(path).exists():
        raise FileNotFoundError(f"Run {notebook} first; missing {path}")


def load(columns):
    """Read requested master columns and apply the configured listing filters."""
    requested = list(dict.fromkeys(columns))
    read_columns = list(dict.fromkeys(requested + FILTER_COLUMNS))
    frame = pd.read_parquet(MASTER, columns=read_columns)
    valid_values = {
        column: sorted(frame[column].dropna().astype(str).unique())
        for column in ("market", "market_building__submarket")
    }
    for column, value, label in (
        ("market", CITY, "city"),
        ("market_building__submarket", NEIGHBORHOOD, "neighborhood"),
    ):
        if value is not None:
            wanted = list(map(str, _as_list(value)))
            valid = valid_values[column]
            unknown = sorted(set(wanted) - set(valid))
            if unknown:
                raise ValueError(f"Unknown {label}(s) {unknown}. Valid values: {valid}")
            before = len(frame)
            frame = frame[frame[column].astype(str).isin(wanted)]
            note(f"{label}: {before:,} → {len(frame):,} rows")
    if START_DATE is not None:
        before = len(frame)
        frame = frame[frame[DATE_COLUMN] >= pd.Timestamp(START_DATE)]
        note(f"start date: {before:,} → {len(frame):,} rows")
    if END_DATE is not None:
        before = len(frame)
        frame = frame[frame[DATE_COLUMN] <= pd.Timestamp(END_DATE)]
        note(f"end date: {before:,} → {len(frame):,} rows")
    return frame[requested]

DB = ROOT / "data" / "accolade.duckdb"
CATALOG = ROOT / "data" / "master_columns.csv"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
con = duckdb.connect(str(DB), read_only=True)
con.execute(f"CREATE TEMP VIEW master AS SELECT * FROM read_parquet('{MASTER.as_posix()}')")
catalog = pd.read_csv(CATALOG)
results = []


def scalar(sql, params=None):
    return con.execute(sql, params).fetchone()[0]


def check(section, name, observed, expected=0):
    results.append({"section": section, "check": name, "expected": expected,
                    "observed": observed, "pass": observed == expected})


def report(section):
    return pd.DataFrame([r for r in results if r["section"] == section]).drop(columns="section")


if any(value is not None for value in (CITY, NEIGHBORHOOD, START_DATE, END_DATE)):
    note("Join checks run on the whole table; filters apply only to the first-30-columns display.")


## 1. Load

In [3]:
cols = [r[0] for r in con.execute("DESCRIBE master").fetchall()]
n_rows = scalar("SELECT count(*) FROM master")
print(f"{MASTER.name}: {n_rows:,} rows x {len(cols)} columns, {MASTER.stat().st_size / 1e6:.0f} MB")
catalog["role"].value_counts().rename("n_columns").to_frame()

master_listing.parquet: 967,661 rows x 378 columns, 207 MB


,n_columns
role,
feature,317
latent,23
label,17
leaky,14
key,4
inferred,3


## 2. First 30 columns

These are the 4 join keys, the 14 `market.unit` columns, and the first 12 `truth.lease_event`
columns. The lease-event columns are mostly `label` (ground truth): they are prediction
targets, so they must not be used as features.

The 59 join checks below always run on the whole table: a join is right or wrong for every row.
Filters apply only to this first-30-columns display.


In [4]:
first30 = cols[:30]
df30 = load(first30)
df30.head(10)

,unit_id,property_id,floor_plan_id,market,market_unit__unit_name,market_unit__floorplan_name,market_unit__bed,market_unit__bath,market_unit__partial_bath,market_unit__floor,market_unit__sqft,market_unit__days_on_market,market_unit__availability,market_unit__enter_market,market_unit__exit_market,market_unit__created_on,market_unit__url,market_unit__status,truth_lease_event__physical_unit_id,truth_lease_event__outcome,truth_lease_event__true_lease_start,truth_lease_event__true_lease_end,truth_lease_event__true_term_months,truth_lease_event__true_rent,truth_lease_event__true_concession_monthly,truth_lease_event__true_effective_rent,truth_lease_event__true_move_in,truth_lease_event__inferred_lease_start,truth_lease_event__inferred_lease_end,truth_lease_event__inferred_term_months
0,9f87b068-L000226,9f87b068-f782-40f3-855e-0e7e587eb4df,1cd70092-0481-4cda-88be-9c5292e28d00,chicago,503,1 Bed 1 Bath,1,1,0,5.0,850.0,31,2022-12-16,2022-12-03,2023-01-03,2022-12-03 06:21:48,https://listings.example/9f87b068/listing/9f87...,closed,9f87b068-503,leased,2023-01-08,2024-01-08,12.0,1240.0,83.67,1156.33,2023-01-08,2023-01-17,2024-01-17,12.0
1,9f87b068-L000349,9f87b068-f782-40f3-855e-0e7e587eb4df,1cd70092-0481-4cda-88be-9c5292e28d00,chicago,1802,1 Bed 1 Bath,1,1,0,18.0,850.0,20,2023-12-22,2023-12-08,2023-12-28,2023-12-10 06:19:44,https://listings.example/9f87b068/listing/9f87...,closed,9f87b068-1802,leased,2023-12-30,2024-06-30,6.0,1340.0,100.75,1239.25,2023-12-30,2024-01-11,2025-01-11,12.0
2,a30f7758-L000163,a30f7758-ddd1-4253-97c0-092044f69c99,a1e1eed8-8a6c-4da9-8a7a-0d82c044f05a,chicago,1105,Amber,2,2,0,NaN,905.0,89,2021-04-23,2021-02-10,2021-05-10,2021-02-10 10:32:00,https://lakeviewliving.example/chicago-01024/l...,closed,a30f7758-1105,leased,2021-05-13,2021-12-13,7.0,1350.0,0.00,1350.00,2021-05-13,2021-05-24,2022-05-24,12.0
3,a4d71e3f-L000543,a4d71e3f-01fc-4706-a50d-48b5a83a70d4,a0f8e955-ef59-4fed-be71-267f365a0cc2,chicago,106,Pine,0,1,0,1.0,540.0,26,2024-05-09,2024-04-19,2024-05-15,2024-04-20 13:05:53,https://lakeviewliving.example/chicago-00530/l...,closed,a4d71e3f-106,leased,2024-06-01,2025-01-01,7.0,1565.0,100.00,1465.00,2024-06-01,2024-05-29,2025-05-29,12.0
4,7d7f3f38-L000484,7d7f3f38-52cb-4537-b9ed-808178cf9d8a,33c6ae4e-c0f7-4b04-8380-e0b0923b404c,dfw,325,1x1 Grand,1,1,0,NaN,785.0,115,2025-01-12,2024-10-30,2025-02-22,2024-10-30 14:25:34,https://lakefrontsenior.example/dfw-00782/list...,closed,7d7f3f38-325,leased,2025-02-25,2026-02-25,12.0,1800.0,0.00,1800.00,2025-02-25,2025-03-08,2026-03-08,12.0
5,7d7f3f38-L000559,7d7f3f38-52cb-4537-b9ed-808178cf9d8a,54980557-93b6-48cd-9a33-d941596beda2,dfw,171,3x2 Grand,2,2,0,NaN,1105.0,25,2025-05-26,2025-03-20,2025-04-14,2025-03-22 07:12:57,https://lakefrontsenior.example/dfw-00782/list...,closed,7d7f3f38-171,leased,2025-05-26,2026-03-26,10.0,2245.0,281.07,1963.93,2025-05-26,2025-04-28,2026-04-28,12.0
6,8220c241-L000060,8220c241-c2a4-4ae1-a8b0-1c5bd248bcb2,3f175bb3-4ec3-4338-9ac1-dd4068c56d44,chicago,<NA>,1x1 Deluxe,1,1,0,NaN,655.0,67,2025-06-13,2025-05-22,2025-07-28,2025-05-24 18:42:08,https://riverbendliving.example/chicago-00970/...,closed,8220c241-207,leased,2025-08-26,2026-07-26,11.0,1320.0,104.17,1215.83,2025-08-26,2025-08-11,2026-08-11,12.0
7,82283740-L000882,82283740-999c-4b70-bc78-05f4e715253b,12525c66-16ec-45bc-981d-0ab96d1b03ea,chicago,2105,2x2 Grand,2,2,0,NaN,955.0,21,2023-02-07,2023-01-23,2023-02-13,2023-01-25 07:21:51,https://lakefrontsenior.example/chicago-00801/...,closed,82283740-2105,leased,2023-02-18,2024-03-18,13.0,1630.0,0.00,1630.00,2023-02-18,2023-02-27,2024-02-27,12.0
8,8666bfc5-L000224,8666bfc5-c6d6-4d7a-82d2-83e97ae2ebfd,53bbf16c-8a6f-4bb3-b9b6-fe759a4ca848,dfw,113,2x2 Grand,1,1,0,NaN,685.0,28,2023-02-02,2023-01-20,2023-02-17,2023-01-22 13:03:31,https://lakefrontresidential.example/dfw-00319...,closed,8666bfc5-113,leased,2023-02-25,2024-02-25,12.0,780.0,71.25,708.75,2023-02-25,2023-03-03,2024-03-03,12.0
9,932872a1-L000584,932872a1-0e05-48ec-aacb-b6c0c2a87d23,fb26fb75-9f38-4

In [5]:
(catalog.set_index("column").loc[first30, ["source_table", "role"]]
 .assign(dtype=df30.dtypes.astype(str),
         null_pct=(df30.isna().mean() * 100).round(2),
         n_unique=df30.nunique()))

,source_table,role,dtype,null_pct,n_unique
column,,,,,
unit_id,market.unit,key,string,0.00,967661
property_id,market.unit,key,string,0.00,3000
floor_plan_id,market.unit,key,string,0.00,22484
market,market.unit,key,string,0.00,3
market_unit__unit_name,market.unit,feature,string,10.78,2643
market_unit__floorplan_name,market.unit,feature,string,0.00,576
market_unit__bed,market.unit,feature,int64,0.00,5
market_unit__bath,market.unit,feature,int64,0.00,2
market_unit__partial_bath,market.unit,feature,int64,0.00,2


## 3. Grain: one row per listing

The master must have exactly the listings in `market.unit`, once each, with the same keys.

In [6]:
n_units = scalar("SELECT count(*) FROM market.unit")
keys = "unit_id, property_id, floor_plan_id, market"
check("grain", "rows == listings in market.unit", n_rows, n_units)
check("grain", "duplicate unit_id", n_rows - scalar("SELECT count(DISTINCT unit_id) FROM master"))
check("grain", "null join keys", scalar(
    "SELECT count(*) FROM master WHERE unit_id IS NULL OR property_id IS NULL "
    "OR floor_plan_id IS NULL OR market IS NULL"))
check("grain", "master keys not in market.unit", scalar(
    f"SELECT count(*) FROM (SELECT {keys} FROM master EXCEPT SELECT {keys} FROM market.unit)"))
check("grain", "market.unit listings not in master", scalar(
    f"SELECT count(*) FROM (SELECT {keys} FROM market.unit EXCEPT SELECT {keys} FROM master)"))
report("grain")

,check,expected,observed,pass
0,rows == listings in market.unit,967661,967661,True
1,duplicate unit_id,0,0,True
2,null join keys,0,0,True
3,master keys not in market.unit,0,0,True
4,market.unit listings not in master,0,0,True


## 4. No fan-out: every right-side join key is unique

If a key appeared twice in a source table, the LEFT JOIN would duplicate the listing.
`market.price_observation` and `market.concession_event` have many rows per listing, but
the build aggregates them by `unit_id` before joining, so they cannot fan out.

In [7]:
JOIN_KEYS = {
    "truth.lease_event": "unit_id",
    "market.floor_plan": "floor_plan_id",
    "market.building": "property_id",
    "truth.building_params": "property_id",
    "market.leasing_activity_daily": 'property_id, "As of Day"',
    "truth.floorplan_market_rent_monthly": "floor_plan_id, month",
    "pms.property": "property_id",
}
for table, key in JOIN_KEYS.items():
    n, n_distinct = con.execute(f"SELECT count(*), count(DISTINCT ({key})) FROM {table}").fetchone()
    check("fan-out", f"duplicate ({key}) in {table}", n - n_distinct)
report("fan-out")

,check,expected,observed,pass
0,duplicate (unit_id) in truth.lease_event,0,0,True
1,duplicate (floor_plan_id) in market.floor_plan,0,0,True
2,duplicate (property_id) in market.building,0,0,True
3,duplicate (property_id) in truth.building_params,0,0,True
4,"duplicate (property_id, ""As of Day"") in market...",0,0,True
5,"duplicate (floor_plan_id, month) in truth.floo...",0,0,True
6,duplicate (property_id) in pms.property,0,0,True


## 5. Coverage: each source filled exactly the listings it matches

For each source, a listing is *matched* if the source has a row for it under the build's
join condition. It is *filled* if any of that source's columns is non-null in the master.
These two must agree row by row: a matched listing with no data means the join dropped it,
and an unmatched listing with data means it picked up someone else's row.

`truth.concession` uses the `market.concession_event` condition because it is aggregated
through that table. `pms_property__is_pms` is left out because it is a true/false flag that
is never NULL; section 6 checks it.

In [8]:
MATCH = {
    "truth.lease_event": "truth.lease_event s WHERE s.unit_id = m.unit_id",
    "market.floor_plan": "market.floor_plan s WHERE s.floor_plan_id = m.floor_plan_id",
    "market.building": "market.building s WHERE s.property_id = m.property_id",
    "truth.building_params": "truth.building_params s WHERE s.property_id = m.property_id",
    "market.price_observation": "market.price_observation s WHERE s.unit_id = m.unit_id",
    "market.concession_event": "market.concession_event s WHERE s.unit_id = m.unit_id",
    "truth.concession": "market.concession_event s WHERE s.unit_id = m.unit_id",
    "market.leasing_activity_daily": ('market.leasing_activity_daily s WHERE s.property_id = m.property_id '
                                      'AND s."As of Day" = m.market_unit__enter_market'),
    "truth.floorplan_market_rent_monthly": ("truth.floorplan_market_rent_monthly s "
                                            "WHERE s.floor_plan_id = m.floor_plan_id "
                                            "AND s.month = date_trunc('month', m.market_unit__enter_market)"),
    "pms.property": "pms.property s WHERE s.property_id = m.property_id",
}
coverage = []
for source, cond in MATCH.items():
    src_cols = catalog.loc[(catalog.source_table == source)
                           & (catalog.column != "pms_property__is_pms"), "column"]
    filled = " OR ".join(f'"{c}" IS NOT NULL' for c in src_cols)
    matched, n_filled, matched_empty, unmatched_filled = con.execute(f"""
        SELECT count(*) FILTER (WHERE matched), count(*) FILTER (WHERE filled),
               count(*) FILTER (WHERE matched AND NOT filled),
               count(*) FILTER (WHERE NOT matched AND filled)
        FROM (SELECT EXISTS (SELECT 1 FROM {cond}) AS matched, ({filled}) AS filled FROM master m)
    """).fetchone()
    check("coverage", f"{source}: matched but empty", matched_empty)
    check("coverage", f"{source}: unmatched but filled", unmatched_filled)
    coverage.append({"source": source, "n_columns": len(src_cols), "matched": matched,
                     "filled": n_filled, "coverage_pct": round(100 * matched / n_rows, 2),
                     "matched_but_empty": matched_empty, "unmatched_but_filled": unmatched_filled})
pd.DataFrame(coverage)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,source,n_columns,matched,filled,coverage_pct,matched_but_empty,unmatched_but_filled
0,truth.lease_event,13,967661,967661,100.00,0,0
1,market.floor_plan,1,967661,967661,100.00,0,0
2,market.building,270,967661,967661,100.00,0,0
3,truth.building_params,23,967661,967661,100.00,0,0
4,market.price_observation,20,967277,967277,99.96,0,0
5,market.concession_event,9,71674,71674,7.41,0,0
6,truth.concession,5,71674,71674,7.41,0,0
7,market.leasing_activity_daily,2,875678,875678,90.49,0,0
8,truth.floorplan_market_rent_monthly,2,953936,953936,98.58,0,0
9,pms.property,14,52141,52141,5.39,0,0


Unit-level concessions (7.4%) and PMS properties (150 of 3,000 buildings) are sparse by
design. The two date-keyed joins are the ones that could silently miss, for example through a
type mismatch or an off-by-one month. Their unmatched listings should all have entered the
market before that source's data starts for the same property or floor plan.

In [9]:
unmatched = con.sql("""
    WITH la AS (SELECT property_id, min("As of Day") AS starts
                FROM market.leasing_activity_daily GROUP BY 1),
         fmr AS (SELECT floor_plan_id, min(month) AS starts
                 FROM truth.floorplan_market_rent_monthly GROUP BY 1)
    SELECT 'market.leasing_activity_daily' AS source, count(*) AS unmatched,
           count(*) FILTER (WHERE m.market_unit__enter_market < la.starts) AS entered_before_data
    FROM master m LEFT JOIN la USING (property_id)
    WHERE m.market_leasing_activity_daily__leased_pct_at_enter IS NULL
      AND m.market_leasing_activity_daily__exposure_pct_at_enter IS NULL
    UNION ALL
    SELECT 'truth.floorplan_market_rent_monthly', count(*),
           count(*) FILTER (WHERE date_trunc('month', m.market_unit__enter_market) < fmr.starts)
    FROM master m LEFT JOIN fmr USING (floor_plan_id)
    WHERE m.truth_floorplan_market_rent__true_market_rent_at_enter IS NULL
      AND m.truth_floorplan_market_rent__true_market_rent_psf_at_enter IS NULL
""").df()
for r in unmatched.itertuples():
    check("coverage", f"{r.source}: unmatched not explained by date window",
          r.unmatched - r.entered_before_data)
unmatched

,source,unmatched,entered_before_data
0,market.leasing_activity_daily,91983,91983
1,truth.floorplan_market_rent_monthly,13725,13725


## 6. Right match: redundant keys agree

Several sources carry keys the join did not use (for example, `truth.lease_event` has its own
`property_id`). If the join linked the right row, those keys must equal the listing's own keys.
Both the market layer and the truth layer flag PMS buildings, so those flags must also agree.

In [10]:
CONSISTENCY = {
    "lease_event.property_id != unit": ("truth.lease_event s USING (unit_id)",
                                           "s.property_id", "m.property_id"),
    "lease_event.floor_plan_id != unit": ("truth.lease_event s USING (unit_id)",
                                             "s.floor_plan_id", "m.floor_plan_id"),
    "lease_event.market != unit": ("truth.lease_event s USING (unit_id)", "s.market", "m.market"),
    "floor_plan.property_id != unit": ("market.floor_plan s USING (floor_plan_id)",
                                          "s.property_id", "m.property_id"),
    "floor_plan.market != unit": ("market.floor_plan s USING (floor_plan_id)", "s.market", "m.market"),
    "building.market != unit": ("market.building s USING (property_id)", "s.market", "m.market"),
    "building_params.market != unit": ("truth.building_params s USING (property_id)", "s.market", "m.market"),
    "price_observation.property_id != unit": ("market.price_observation s USING (unit_id)",
                                                "s.property_id", "m.property_id"),
    "concession_event.property_id != unit": ("market.concession_event s USING (unit_id)",
                                               "s.property_id", "m.property_id"),
    "floorplan_market_rent.property_id != unit": (
        "truth.floorplan_market_rent_monthly s ON s.floor_plan_id = m.floor_plan_id "
        "AND s.month = date_trunc('month', m.market_unit__enter_market)", "s.property_id", "m.property_id"),
}
for name, (join, a, b) in CONSISTENCY.items():
    check("keys", name, scalar(f"SELECT count(*) FROM master m JOIN {join} WHERE {a} IS DISTINCT FROM {b}"))
check("keys", "pms_property__is_pms != truth_building_params__is_pms", scalar(
    "SELECT count(*) FROM master WHERE pms_property__is_pms IS DISTINCT FROM truth_building_params__is_pms"))
report("keys")

,check,expected,observed,pass
0,lease_event.property_id != unit,0,0,True
1,lease_event.floor_plan_id != unit,0,0,True
2,lease_event.market != unit,0,0,True
3,floor_plan.property_id != unit,0,0,True
4,floor_plan.market != unit,0,0,True
5,building.market != unit,0,0,True
6,building_params.market != unit,0,0,True
7,price_observation.property_id != unit,0,0,True
8,concession_event.property_id != unit,0,0,True
9,floorplan_market_rent.property_id != unit,0,0,True


## 7. Right values: re-derive columns straight from the sources

For one column per source, recompute the value from the raw table independently of the build
SQL and compare it with the master on every row. Aggregates are recomputed with their own
`GROUP BY`. Values are copied, not transformed, so they must match exactly.

In [11]:
VALUES = {
    "market_unit__enter_market": "SELECT unit_id, enter_market AS v FROM market.unit",
    "truth_lease_event__true_rent": "SELECT unit_id, true_rent AS v FROM truth.lease_event",
    "truth_lease_event__outcome": "SELECT unit_id, outcome AS v FROM truth.lease_event",
    "market_floor_plan__unit_count": (
        "SELECT u.unit_id, fp.unit_count AS v FROM market.unit u "
        "JOIN market.floor_plan fp ON fp.floor_plan_id = u.floor_plan_id"),
    "market_building__year_built": (
        "SELECT u.unit_id, b.year_built AS v FROM market.unit u "
        "JOIN market.building b ON b.property_id = u.property_id"),
    "truth_building_params__quality_true": (
        "SELECT u.unit_id, bp.quality_true AS v FROM market.unit u "
        "JOIN truth.building_params bp ON bp.property_id = u.property_id"),
    "market_price_observation__n_windows": (
        "SELECT unit_id, count(*) AS v FROM market.price_observation GROUP BY unit_id"),
    "market_price_observation__max_asking_rent": (
        "SELECT unit_id, max(asking_rent) AS v FROM market.price_observation GROUP BY unit_id"),
    "market_concession_event__n_events": (
        "SELECT unit_id, count(*) AS v FROM market.concession_event "
        "WHERE unit_id IS NOT NULL GROUP BY unit_id"),
    "market_leasing_activity_daily__leased_pct_at_enter": (
        "SELECT u.unit_id, la.leased_percentage AS v FROM market.unit u "
        'JOIN market.leasing_activity_daily la ON la.property_id = u.property_id '
        'AND la."As of Day" = u.enter_market'),
    "truth_floorplan_market_rent__true_market_rent_at_enter": (
        "SELECT u.unit_id, fmr.true_market_rent AS v FROM market.unit u "
        "JOIN truth.floorplan_market_rent_monthly fmr ON fmr.floor_plan_id = u.floor_plan_id "
        "AND fmr.month = date_trunc('month', u.enter_market)"),
    "pms_property__sid": (
        "SELECT u.unit_id, p.sid AS v FROM market.unit u "
        "JOIN pms.property p ON p.property_id = u.property_id"),
}
values = []
for col, sub in VALUES.items():
    mismatches, compared = con.execute(
        f'SELECT count(*) FILTER (WHERE m."{col}" IS DISTINCT FROM s.v), count(s.v) '
        f"FROM master m LEFT JOIN ({sub}) s USING (unit_id)").fetchone()
    check("values", f"{col} differs from source", mismatches)
    values.append({"column": col, "rows_compared": compared, "mismatches": mismatches})
pd.DataFrame(values)

,column,rows_compared,mismatches
0,market_unit__enter_market,967661,0
1,truth_lease_event__true_rent,931738,0
2,truth_lease_event__outcome,967661,0
3,market_floor_plan__unit_count,967661,0
4,market_building__year_built,863321,0
5,truth_building_params__quality_true,967661,0
6,market_price_observation__n_windows,967277,0
7,market_price_observation__max_asking_rent,967277,0
8,market_concession_event__n_events,71674,0
9,market_leasing_activity_daily__leased_pct_at_e...,875678,0


### Trace one listing

The same comparison for a single listing that touches every source, so you can see the values side by side.

In [12]:
uid = scalar("""
    SELECT unit_id FROM master
    WHERE truth_lease_event__outcome = 'leased' AND pms_property__is_pms
      AND market_concession_event__n_events IS NOT NULL
      AND market_leasing_activity_daily__leased_pct_at_enter IS NOT NULL
    ORDER BY hash(unit_id) LIMIT 1
""")
print("unit_id:", uid)
pd.DataFrame([
    {"column": col,
     "master": scalar(f'SELECT "{col}" FROM master WHERE unit_id = ?', [uid]),
     "source": scalar(f"SELECT v FROM ({sub}) WHERE unit_id = ?", [uid])}
    for col, sub in VALUES.items()
])

unit_id: c4459412-L000081


,column,master,source
0,market_unit__enter_market,2021-05-06 00:00:00,2021-05-06
1,truth_lease_event__true_rent,1395.0,1395.0
2,truth_lease_event__outcome,leased,leased
3,market_floor_plan__unit_count,38,38
4,market_building__year_built,2007.0,2007
5,truth_building_params__quality_true,0.4278,0.4278
6,market_price_observation__n_windows,2.0,2
7,market_price_observation__max_asking_rent,1395.0,1395.0
8,market_concession_event__n_events,1.0,1
9,market_leasing_activity_daily__leased_pct_at_e...,0.9826,0.9826


## 8. Column catalog matches the parquet

In [13]:
check("catalog", "catalog columns != parquet columns (name and order)",
      int(catalog["column"].tolist() != cols))
check("catalog", "columns with an unknown role",
      int((~catalog["role"].isin(["key", "feature", "label", "leaky", "latent", "inferred"])).sum()))
report("catalog")

,check,expected,observed,pass
0,catalog columns != parquet columns (name and o...,0,0,True
1,columns with an unknown role,0,0,True


## Summary

In [14]:
summary = pd.DataFrame(results, columns=["section", "check", "expected", "observed", "pass"])
print(f"{summary['pass'].sum()} / {len(summary)} checks passed")
summary_path = OUTPUTS / "01_phase_01_master_table_checks.csv"
summary.to_csv(summary_path, index=False)
done(f"Wrote {summary_path.relative_to(ROOT)}")
failed = summary[~summary["pass"]]
assert failed.empty, f"failed checks:\n{failed.to_string()}"
summary.groupby("section", sort=False)["pass"].agg(checks="size", passed="sum")

59 / 59 checks passed
  ✓ Wrote outputs/01_phase_01_master_table_checks.csv


,checks,passed
section,,
grain,5,5
fan-out,7,7
coverage,22,22
keys,11,11
values,12,12
catalog,2,2
